# Chapter 5 &mdash; Verify by Constructing Twice

**Concept 4 of the Chapter 5 decomposition:** *Verification by Construction Twice: Minimal DFA Uniqueness and `iso_dfa`*

Design from two perspectives, minimize both, and check isomorphism &mdash; Myhill&ndash;Nerode guarantees they must match.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Verification-By-Construction-Twice/Concept-Verification-By-Construction-Twice.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The strongest cheap check available: **build the DFA twice from two different
perspectives**, minimize both, and ask whether they are **isomorphic**.

Myhill&ndash;Nerode (Chapter 6) says the minimal DFA for a regular language is **unique up
to renaming**. So:

* two correct designs $\Rightarrow$ `iso_dfa(min_dfa(A), min_dfa(B))` is `True`;
* `False` $\Rightarrow$ **at least one design is wrong**, and you must find out which.

This turns "I think it's right" into a decidable question.

## 2. Definitions

### Perspective 1: track the parity pair directly

In [ ]:
A = md2mc('''DFA
IF : 0 -> P0
IF : 1 -> P1
P0 : 0 -> IF
P0 : 1 -> P01
P1 : 0 -> P01
P1 : 1 -> IF
P01: 0 -> P1
P01: 1 -> P0
''')

### Perspective 2: even length, then even zeros &mdash; a different decomposition

In [ ]:
B = md2mc('''DFA
IF  : 0 -> Od0     !! odd length, odd 0s
IF  : 1 -> Od1     !! odd length, even 0s
Od0 : 0 -> IF
Od0 : 1 -> Ev2
Od1 : 0 -> Ev2
Od1 : 1 -> IF
Ev2 : 0 -> Od1
Ev2 : 1 -> Od0
''')

### A deliberately WRONG third design, to see the check fire

In [ ]:
Bad = md2mc('''DFA
IF : 0 -> P0
IF : 1 -> IF     !! BUG: 1s ignored
P0 : 0 -> IF
P0 : 1 -> P0
''')

## 3. Tests

Two correct designs minimize to isomorphic machines.

In [ ]:
mA, mB = min_dfa(A), min_dfa(B)
print("|Q| after minimization : A=%d  B=%d" % (len(mA["Q"]), len(mB["Q"])))
print("isomorphic? ", iso_dfa(mA, mB))
assert iso_dfa(mA, mB)
print("\nMyhill-Nerode: the minimal DFA is unique, so this HAD to hold.")

The wrong design is caught &mdash; and `langeq_dfa` produces a counterexample string.

In [ ]:
print("Bad isomorphic to A? ", iso_dfa(min_dfa(Bad), mA))
print("Bad language-equal?  ", langeq_dfa(Bad, A))
assert not langeq_dfa(Bad, A)
from itertools import product
wit = next(''.join(p) for k in range(6) for p in product('01', repeat=k)
           if accepts_dfa(Bad, ''.join(p)) != accepts_dfa(A, ''.join(p)))
print("shortest witness :", repr(wit),
      " A says", accepts_dfa(A, wit), " Bad says", accepts_dfa(Bad, wit))

Note the two checks answer different questions.

In [ ]:
print("langeq_dfa : same LANGUAGE (works on any two DFA)")
print("iso_dfa    : same SHAPE up to renaming (meaningful after min_dfa)")
print()
print("A vs B before minimizing: langeq=%s iso=%s"
      % (langeq_dfa(A, B), iso_dfa(A, B)))

## 4. Exercises


1. Design a third perspective on the same language and check all three.
2. Why is `iso_dfa` on *non*-minimal DFA a weak test?
3. When `iso_dfa` fails, how do you tell which of the two designs is wrong?

In [ ]:
# Your work for the exercises above.